# 02 - Data Preparation & Modelling
**Capstone Project - Machine Learning UAS**
Nama: Laily Muthia N | NIM: A11.2024.15618

CRISP-DM Tahap 3 (Data Preparation) & Tahap 4 (Modelling)

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np
import joblib
from data_preprocessing import run_pipeline, clean_data, feature_engineering, load_data
import warnings
warnings.filterwarnings("ignore")

import os
os.chdir("..")  # agar path relatif (data/, models/) konsisten dengan script src/

## 1. Data Cleaning
Menangani nilai 0 pada `Cholesterol` dan `RestingBP` yang secara klinis tidak wajar,
diimputasi dengan median per kelas target agar distribusi antar kelas tetap terjaga.

In [2]:
df_raw = load_data()
df_clean = clean_data(df_raw)

print("Sebelum cleaning - Cholesterol=0:", (df_raw.Cholesterol==0).sum())
print("Setelah cleaning  - Cholesterol=0:", (df_clean.Cholesterol==0).sum())
df_clean.describe().T[["min","mean","max"]]

Sebelum cleaning - Cholesterol=0: 172
Setelah cleaning  - Cholesterol=0: 0


,min,mean,max
Age,28.0,53.510893,77.0
RestingBP,80.0,132.540305,200.0
Cholesterol,85.0,244.586057,603.0
FastingBS,0.0,0.233115,1.0
MaxHR,60.0,136.809368,202.0
Oldpeak,-2.6,0.887364,6.2
HeartDisease,0.0,0.553377,1.0


## 2. Feature Engineering
Menambahkan fitur turunan: `AgeGroup` (kelompok usia) dan `HR_Reserve` (cadangan detak jantung,
selisih estimasi HR maksimal teoritis dengan MaxHR aktual pasien).

In [3]:
df_fe = feature_engineering(df_clean)
df_fe[["Age","AgeGroup","MaxHR","HR_Reserve"]].head()

,Age,AgeGroup,MaxHR,HR_Reserve
0,40,<=40,172,8
1,49,41-50,156,15
2,37,<=40,98,85
3,48,41-50,108,64
4,54,51-60,122,44


## 3. Encoding, Scaling, & Split Data
- **Scaling**: StandardScaler untuk fitur numerik
- **Encoding**: One-Hot Encoding untuk fitur kategorikal
- **Split**: 70% train, 10% validation, 20% test, distratifikasi pada target

In [4]:
result = run_pipeline(save=True)
print("Train:", result["X_train"].shape)
print("Val  :", result["X_val"].shape)
print("Test :", result["X_test"].shape)
print("\nProporsi target di train:")
print(result["y_train"].value_counts(normalize=True))

Train: (642, 11)
Val  : (92, 11)
Test : (184, 11)

Proporsi target di train:
HeartDisease
1    0.55296
0    0.44704
Name: proportion, dtype: float64


## 4. Modelling: Random Forest, XGBoost, SVM
Ketiga model dilatih menggunakan **Pipeline** (preprocessing + classifier) dan
di-tuning dengan **GridSearchCV** (5-fold cross-validation, scoring=F1).

In [5]:
from train_model import train_all

fitted_models, results = train_all(
    result["X_train"], result["y_train"],
    result["X_val"], result["y_val"]
)


[TRAINING] RandomForest ...
  Best CV F1   : 0.8782
  Val F1       : 0.8317
  Val Accuracy : 0.8152
  Best Params  : {'clf__max_depth': 5, 'clf__min_samples_leaf': 2, 'clf__min_samples_split': 5, 'clf__n_estimators': 300}

[TRAINING] XGBoost ...
  Best CV F1   : 0.8868
  Val F1       : 0.8868
  Val Accuracy : 0.8696
  Best Params  : {'clf__learning_rate': 0.1, 'clf__max_depth': 3, 'clf__n_estimators': 100, 'clf__subsample': 0.8}

[TRAINING] SVM ...
  Best CV F1   : 0.8779
  Val F1       : 0.8431
  Val Accuracy : 0.8261
  Best Params  : {'clf__C': 1, 'clf__gamma': 'auto', 'clf__kernel': 'rbf'}


In [6]:
results_df = pd.DataFrame(results).T
results_df

,best_params,cv_best_f1,val_f1,val_accuracy
RandomForest,"{'clf__max_depth': 5, 'clf__min_samples_leaf':...",0.878214,0.831683,0.815217
XGBoost,"{'clf__learning_rate': 0.1, 'clf__max_depth': ...",0.886768,0.886792,0.869565
SVM,"{'clf__C': 1, 'clf__gamma': 'auto', 'clf__kern...",0.877883,0.843137,0.826087


## 5. Pemilihan Model Terbaik
Model terbaik dipilih berdasarkan **F1-Score tertinggi pada data validasi**,
karena metrik ini menyeimbangkan Precision dan Recall — penting dalam konteks
medis di mana False Negative (pasien sakit terdiagnosis normal) sangat berisiko.

In [7]:
best_name = max(results, key=lambda k: results[k]["val_f1"])
best_model = fitted_models[best_name]
print(f"Model terbaik: {best_name}")
print(f"Val F1-Score : {results[best_name]['val_f1']:.4f}")
print(f"Best Params  : {results[best_name]['best_params']}")

for name, model in fitted_models.items():
    joblib.dump(model, f"models/{name}.pkl")
joblib.dump(best_model, "models/best_model.pkl")
with open("models/best_model_name.txt","w") as f:
    f.write(best_name)
print("\nModel tersimpan di folder models/")

Model terbaik: XGBoost
Val F1-Score : 0.8868
Best Params  : {'clf__learning_rate': 0.1, 'clf__max_depth': 3, 'clf__n_estimators': 100, 'clf__subsample': 0.8}

Model tersimpan di folder models/
